In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F
# reading data from silver

MPS = spark.read.format("delta").load(f"{Silver_Base}/Silver_MPS_minor_crime")
FLY = spark.read.format("delta").load(f"{Silver_Base}/silver_fly_tipping")
CST = spark.read.format("delta").load(f"{Silver_Base}/silver_community_strength")

In [0]:

def Clean_Joinkeys (df):
    return (
        df
        .withColumn("area", F.lower(F.trim(F.col("area"))))
        .withColumn("area", F.regexp_replace("area", " council", ""))
        .withColumn("area", F.regexp_replace("area", "&", "and"))
        .withColumn("area", F.regexp_replace("area", r"\s+", " "))
        .withColumn("financial_year", F.trim(F.col("financial_year")))
        .withColumn("financial_year", F.regexp_replace("financial_year", "-", "/"))
        .withColumn("financial_year", F.regexp_replace("financial_year", "_", "/"))
    )

MPS = Clean_Joinkeys(MPS)
FLY = Clean_Joinkeys(FLY)

CST = (
    CST
    .withColumn("area", F.lower(F.trim(F.col("area"))))
    .withColumn("area", F.regexp_replace("area", " council", ""))
    .withColumn("area", F.regexp_replace("area", "&", "and"))
    .withColumn("area", F.regexp_replace("area", r"\s+", " "))
)

print("MPS crime no of rows :", MPS.count())
print("FLY tipping no of rows:", FLY.count())
print("CST no of rows:", CST.count())

In [0]:
#joining tables
MPS_FLY = MPS.join(
    FLY,
    on=["area", "financial_year"],
    how="inner"
)

print("MPS and FLY joined rows:", MPS_FLY.count())
display(MPS_FLY.orderBy("area", "financial_year").limit(15))

In [0]:
# joining with community strength
# Note: CST dataset contains one time period only (no financial_year column)
# Community strength scores are therefore treated as static borough-level 
# indicators and joined on area only. 
Gold = (
    MPS_FLY
    .join(CST, on="area", how="inner")
    .dropna()
)

print("Rows in Gold:", Gold.count())
display(Gold.orderBy("area", "financial_year").limit(10))

In [0]:
# gold schema 
print("Gold schema ")
Gold.printSchema()
print("Gold row count ", Gold.count())
print("Gold columns ", Gold.columns)

In [0]:
#Saving 
Gold.write.mode("overwrite").format("delta").save(
    f"{Gold_base}/Gold_Borough_Year"
)

Gold.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{Gold_base}/Export_tab"
)

print("Gold file is successfully saved")